# A first look at the prepared AML transactions

A few simple plots from `data/features.csv`. This file is made by the preparation script and is not stored in Git. I am looking only at the fixed USD ACH subset, not all of IBM's dataset. The label marks simulated laundering activity.

From the repository root, run `python -m src.download_data` and `python -m src.build_dataset --input data/raw/HI-Small_Trans.csv --output data/features.csv` first. Install `matplotlib` if needed. Run this notebook from either the repository root or the `notebooks/` folder.


In [ ]:
import csv
from collections import Counter, defaultdict
from datetime import datetime
from math import expm1
from pathlib import Path

import matplotlib.pyplot as plt

path = Path("data/features.csv")
if not path.exists():
    path = Path("../data/features.csv")
if not path.exists():
    raise FileNotFoundError("Build data/features.csv using the commands in the README first.")

with path.open(newline="", encoding="utf-8") as handle:
    rows = list(csv.DictReader(handle))

labels = Counter(int(row["label"]) for row in rows)
print(f"Prepared transactions: {len(rows):,}")
print(f"Simulated label-1: {labels[1]:,} ({labels[1] / len(rows):.2%})")
print("Split sizes:", dict(Counter(row["split"] for row in rows)))


## 1. How uneven are the labels?

Most transactions have label 0. This is why accuracy alone would be a poor way to judge the models.


In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 3.4))
ax.bar(["Label 0", "Label 1"], [labels[0], labels[1]], color=["#7d9997", "#16747a"])
ax.set_ylabel("Transactions")
ax.set_title("Labels in the prepared subset")
ax.bar_label(ax.containers[0], fmt="{:,.0f}")
plt.tight_layout()
plt.show()


## 2. Transaction amounts

The feature CSV stores `log(1 + amount)`, so I undo that transformation for this plot. A log-scale horizontal axis helps show the spread without hiding the small payments.


In [ ]:
amounts = [expm1(float(row["log_amount"])) for row in rows]
fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.hist(amounts, bins=60, color="#16747a")
ax.set_xscale("log")
ax.set_xlabel("Current payment (USD, log scale)")
ax.set_ylabel("Transactions")
ax.set_title("Payment amounts in the prepared subset")
plt.tight_layout()
plt.show()


## 3. Activity by day

The date split matters here: the first six days are for training, then two for validation, then two for testing.


In [ ]:
daily = defaultdict(Counter)
for row in rows:
    day = datetime.fromisoformat(row["timestamp"].replace("Z", "+00:00")).date()
    daily[day][int(row["label"])] += 1
days = sorted(daily)
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
axes[0].bar(days, [sum(daily[d].values()) for d in days], color="#7d9997")
axes[0].set_ylabel("All rows")
axes[1].bar(days, [daily[d][1] for d in days], color="#16747a")
axes[1].set_ylabel("Label 1")
axes[1].set_xlabel("September 2022")
fig.suptitle("Selected transactions over time")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


## 4. Recent outgoing history in the training period

I compare the number of prior outgoing payments for each label using training rows only. This is a description of the selected subset, not a claim that the feature causes laundering or works on real customers.


In [ ]:
train = [row for row in rows if row["split"] == "train"]
counts = [[min(int(float(row["sender_count_24h"])), 5) for row in train if int(row["label"]) == label] for label in (0, 1)]
fig, ax = plt.subplots(figsize=(7, 3.6))
for values, label, color in zip(counts, ("Label 0", "Label 1"), ("#7d9997", "#16747a")):
    hist = Counter(values)
    ax.plot(range(6), [hist[i] / len(values) for i in range(6)], marker="o", label=label, color=color)
ax.set_xticks(range(6), ["0", "1", "2", "3", "4", "5+"])
ax.set_xlabel("Earlier outgoing payments in 24 hours")
ax.set_ylabel("Share within label")
ax.set_title("Training rows only")
ax.legend()
plt.tight_layout()
plt.show()


## What I noticed

The labels are rare, and many transactions have no recent history within this filtered subset. These plots helped me understand why I should evaluate rankings and review capacity instead of just accuracy. See `results/README.md` for the fixed model comparison. I have left notebook outputs uncommitted so this notebook does not suggest a separate data source or a new experiment.
